## Step 0: Import libraries

In [1]:
import numpy as np
import pandas as pd

## Step 1: Create a large synthetic dataset (500,000 rows)

In [2]:
rng = np.random.default_rng(42)
n = 500_000
df = pd.DataFrame({
    "transaction_id": np.arange(1, n + 1, dtype="int64"),
    "customer_id": rng.integers(1, 50_000, n),
    "store_id": rng.integers(1, 200, n),
    "quantity": rng.integers(1, 20, n),
    "amount": np.round(rng.gamma(2.0, 60.0, n), 2),
    "category": rng.choice(["Electronics", "Clothing", "Groceries", "Furniture", "Books", "Toys"], n).astype(object),
    "payment_method": rng.choice(["Credit Card", "Debit Card", "UPI", "Cash", "Net Banking"], n).astype(object),
    "city": rng.choice(["Surat", "Ahmedabad", "Mumbai", "Pune", "Vadodara", "Rajkot", "Delhi"], n).astype(object),
})
df.head()

,transaction_id,customer_id,store_id,quantity,amount,category,payment_method,city
0,1,4463,15,7,77.74,Clothing,Net Banking,Pune
1,2,38698,162,15,89.93,Furniture,Net Banking,Ahmedabad
2,3,32728,194,1,66.39,Furniture,Cash,Ahmedabad
3,4,21944,74,13,224.99,Books,Credit Card,Rajkot
4,5,21651,143,19,91.78,Books,Cash,Pune


## Step 2: Baseline memory profile – `memory_usage(deep=True)`

In [3]:
def mb(frame):
    """Total deep memory of a DataFrame in MB."""
    return frame.memory_usage(deep=True).sum() / 1024 ** 2

baseline = mb(df)
print(df.dtypes, "\n")
print("Memory per column (MB):")
print((df.memory_usage(deep=True) / 1024 ** 2).round(3))
print(f"\nBASELINE TOTAL: {baseline:.2f} MB")

transaction_id      int64
customer_id         int64
store_id            int64
quantity            int64
amount            float64
category              str
payment_method        str
city                  str
dtype: object 

Memory per column (MB):
Index              0.000
transaction_id     3.815
customer_id        3.815
store_id           3.815
quantity           3.815
amount             3.815
category          27.020
payment_method    27.083
city              26.293
dtype: float64

BASELINE TOTAL: 99.47 MB


## Step 3: Downcast integer columns (int64 → int32 / int16 / int8)

In [4]:
opt = df.copy()
for col in opt.select_dtypes(include="integer").columns:
    opt[col] = pd.to_numeric(opt[col], downcast="integer")

after_int = mb(opt)
print(opt.select_dtypes(include="integer").dtypes.to_dict())
print(f"After integer downcast: {after_int:.2f} MB")

{'transaction_id': dtype('int32'), 'customer_id': dtype('int32'), 'store_id': dtype('int16'), 'quantity': dtype('int8')}
After integer downcast: 89.46 MB


## Step 4: Downcast float columns (float64 → float32)

In [5]:
for col in opt.select_dtypes(include="float").columns:
    opt[col] = pd.to_numeric(opt[col], downcast="float")

after_float = mb(opt)
print(opt.select_dtypes(include="float").dtypes.to_dict())
print(f"After float downcast: {after_float:.2f} MB")

{'amount': dtype('float32')}
After float downcast: 87.55 MB


## Step 5: Convert repetitive string columns to `category`

In [6]:
for col in opt.select_dtypes(include=["object", "string"]).columns:
    if opt[col].nunique() / len(opt) < 0.5:       # low-cardinality columns only
        opt[col] = opt[col].astype("category")

after_cat = mb(opt)
print(opt.dtypes)
print(f"\nAfter category conversion: {after_cat:.2f} MB")

transaction_id       int32
customer_id          int32
store_id             int16
quantity              int8
amount             float32
category          category
payment_method    category
city              category
dtype: object

After category conversion: 8.58 MB


## Step 6: Final report – % reduction in RAM footprint

In [7]:
reduction = (baseline - after_cat) / baseline * 100
print("=" * 45)
print(f"Baseline  : {baseline:8.2f} MB")
print(f"Optimised : {after_cat:8.2f} MB")
print(f"REDUCTION : {reduction:8.2f} %")
print("=" * 45)

Baseline  :    99.47 MB
Optimised :     8.58 MB
REDUCTION :    91.37 %
